In [ ]:
# Shared project setup for imports and file locations
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

DATA_DIR = PROJECT_ROOT / 'data'
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'
FIGURES_DIR = PROJECT_ROOT / 'figures'

def resolve_path(path):
    candidate = Path(path)
    if candidate.exists():
        return candidate
    text = str(path).replace('\\', '/')
    name = Path(text).name
    special = {
        ARTIFACTS_DIR / 'controls' / 'positive_controls.pkl': ARTIFACTS_DIR / 'controls' / 'positive_controls.pkl',
        ARTIFACTS_DIR / 'controls' / 'negative_controls.pkl': ARTIFACTS_DIR / 'controls' / 'negative_controls.pkl',
        'Ten_positive_controls_1119.pkl': ARTIFACTS_DIR / 'controls' / 'positive_controls.pkl',
        'Ten_negative_controls_1119.pkl': ARTIFACTS_DIR / 'controls' / 'negative_controls.pkl',
        DATA_DIR / 'fcg.txt': DATA_DIR / 'fcg.txt',
    }
    if name in special:
        return special[name]
    matches = [p for p in PROJECT_ROOT.rglob(name) if '.ipynb_checkpoints' not in p.parts and '.git' not in p.parts]
    if len(matches) == 1:
        return matches[0]
    if (text.startswith('/Users/') or text.startswith('/home/') or ':\\' in text) and '.' not in name:
        return PROJECT_ROOT
    return candidate

from pdm_learn.preprocessing import build_density_map, density_centers, densitymap, drop_nan, extract, mut_trim, normalize, trim, trim_pairs
from pdm_learn.modeling import KFold_PR, LOOCV, LOOCV_grouped_plot, area_table, core_predict, heatmap, importance_test, ks_pvalue
from pdm_learn.simulation import eps, partition


In [1]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
from bisect import bisect_left

In [2]:
# Shared helper functions now live in src/pdm_learn.
# See the project setup cell at the top of this notebook for imports.


In [3]:
# Shared helper functions now live in src/pdm_learn.
# See the project setup cell at the top of this notebook for imports.


In [ ]:
def get_std(mat1, mat2):
    return np.nanstd(mat1), np.nanstd(mat2);

def avg_std(mat1, mat2):
    return math.sqrt((np.nanvar(mat1)+np.nanvar(mat2))/2)

In [9]:
# Shared helper functions now live in src/pdm_learn.
# See the project setup cell at the top of this notebook for imports.


In [10]:
# Shared helper functions now live in src/pdm_learn.
# See the project setup cell at the top of this notebook for imports.


In [11]:
# Shared helper functions now live in src/pdm_learn.
# See the project setup cell at the top of this notebook for imports.


In [12]:
def build(df1, df2, *args):
    key = {key: i for i, key in enumerate(dataset.iloc[:, 0])}
    
    if 'mut' in args:
        temp = pd.DataFrame(index=range(len(genes)), 
                            columns = [df1.name+'.'+df2.name+'.'+value for value in map(str, range(2*7))])
        df1_t, df2_t = mut_trim(df1, df2)
        if 'cn' in args:
            for gene_name in df2_t.iloc[:,0]:
                x = extract(df1_t, gene_name, cutoff=False)
                y = extract(df2_t, gene_name, cutoff=False)
                x, y = drop_nan(x, y)
                mat = densitymap(x, y, [0,1], [0,1,2,3,4,6,8], xdiscrete=True, ydiscrete=True)
                temp.iloc[key[gene_name]] = mat.flatten()
        else:
            df2_t.iloc[:,1:] = normalize(df2_t.iloc[:,1:])
            std = np.nanstd(df2_t.iloc[:,1:])
            for gene_name in df2_t.iloc[:,0]:
                x = extract(df1_t, gene_name, cutoff=False)
                y, yd = extract(df2_t, gene_name, cutoff=True, std=std, max=7, density_center=True, boxes=7)
                x, y = drop_nan(x, y)
                mat = densitymap(x, y, [0, 1], yd, xdiscrete=True, sigma=std)
                temp.iloc[key[gene_name]] = mat.flatten()
    else:
        temp = pd.DataFrame(index=range(len(genes)), 
                            columns = [df1.name+'.'+df2.name+'.'+value for value in map(str, range(7*7))])
        df1_t, df2_t = trim(df1, df2)
        if'cn' in args:
            df1_t.iloc[:,1:] = normalize(df1_t.iloc[:,1:])
            std = np.nanstd(df1_t.iloc[:,1:])
            for gene_name in df1_t.iloc[:,0]:
                x, xd = extract(df1_t, gene_name, cutoff=True, std=std, max=7, density_center=True, boxes=7)
                y = extract(df2_t, gene_name, cutoff=False)
                x, y = drop_nan(x, y)
                mat = densitymap(x,y,xd,[0,1,2,3,4,6,8], ydiscrete=True, sigma=std)
                temp.iloc[key[gene_name]] = mat.flatten()
        else:
            df1_t.iloc[:,1:] = normalize(df1_t.iloc[:,1:])
            df2_t.iloc[:,1:] = normalize(df2_t.iloc[:,1:])
            xstd = np.nanstd(df1_t.iloc[:,1:].to_numpy())
            ystd = np.nanstd(df2_t.iloc[:,1:].to_numpy())
            avgstd = avg_std(df1_t.iloc[:,1:].to_numpy(), df2_t.iloc[:,1:].to_numpy())
            for gene_name in df1_t.iloc[:,0]:
                x, xd = extract(df1_t, gene_name, cutoff=True, std=xstd, max=7, density_center=True, boxes=7)
                y, yd = extract(df2_t, gene_name, cutoff=True, std=ystd, max=7, density_center=True, boxes=7)
                x, y = drop_nan(x, y)
                mat = densitymap(x,y,xd,yd,sigma=avgstd)
                temp.iloc[key[gene_name]] = mat.flatten()
    
    temp += 1/len(df1_t.columns)
    temp = temp.applymap(np.log)
    
    return temp

In [13]:
gene_exp = pd.read_csv(r"C:\Users\justi\Coding\Coding_Project\Han Xu\DepMap_data\CCLE_gene_expression_trimmed_Wei.csv")
gene_exp.name = 'gene_exp'

copy_num = pd.read_csv(r"C:\Users\justi\Coding\Coding_Project\Han Xu\DepMap_data\CCLE_gene_cn_trimmed_Wei.csv")
copy_num.iloc[:,1:] *= 2
def take_closest(myList, myNumber):
    pos = bisect_left(myList, myNumber)
    if pos == 0:
        return myList[0]
    if pos == len(myList):
        return myList[-1]
    before = myList[pos - 1]
    after = myList[pos]
    if after - myNumber < myNumber - before:
        return after
    else:
        return before 
# List of values to compare for each element in the DataFrame
values_to_compare = [0, 1, 2, 3, 4, 6, 8]
# Apply the take_closest function to every value in the DataFrame
copy_num.iloc[:,1:] = copy_num.iloc[:,1:].map(lambda x: take_closest(values_to_compare, x))
copy_num.name = 'copy_num'

shRNA = pd.read_csv(r"C:\Users\justi\Coding\Coding_Project\Han Xu\DepMap_data\shRNA_Broad_Trimmed_Wei.csv")
shRNA.name = 'shRNA'

gene_mut = pd.read_csv(r"C:\Users\justi\Coding\Coding_Project\Han Xu\DepMap_data\CCLE_gene_mutation_trimmed_Wei.csv")
gene_mut.drop(columns=gene_mut.columns[0],inplace=True)
gene_mut.name = 'gene_mut'

CRISPR = pd.read_csv(r"C:\Users\justi\Coding\Coding_Project\Han Xu\DepMap_data\Avana_gene_effect_20Q3_Trimmed_Wei.csv")
CRISPR.name = 'CRISPR'

C:\Users\justi\AppData\Local\Temp\ipykernel_19128\1847578315.py:27: DtypeWarning: Columns (20,30,31) have mixed types. Specify dtype option on import or set low_memory=False.
  gene_mut = pd.read_csv(r"C:\Users\justi\Coding\Coding_Project\Han Xu\DepMap_data\CCLE_gene_mutation_trimmed_Wei.csv")


In [82]:
genes = sorted(set(gene_exp.iloc[:,0].str.strip()) | 
               set(copy_num.iloc[:,0].str.strip()) | 
               set(shRNA.iloc[:,0].str.strip()) | 
               set(CRISPR.iloc[:,0].str.strip()))
dataset = pd.DataFrame({'gene name':genes})
dataset = pd.concat([dataset, build(gene_exp, copy_num, 'cn')], axis=1)
dataset = pd.concat([dataset, build(gene_exp, shRNA)], axis=1)
dataset = pd.concat([dataset, build(gene_mut, gene_exp, 'mut')], axis=1)
dataset = pd.concat([dataset, build(gene_exp, CRISPR)], axis=1)
dataset = pd.concat([dataset, build(shRNA, copy_num, 'cn')], axis=1)
dataset = pd.concat([dataset, build(gene_mut, copy_num, 'cn', 'mut')], axis=1)
dataset = pd.concat([dataset, build(CRISPR, copy_num, 'cn')], axis=1)
dataset = pd.concat([dataset, build(gene_mut, shRNA, 'mut')], axis=1)
dataset = pd.concat([dataset, build(shRNA, CRISPR)], axis=1)
dataset = pd.concat([dataset, build(gene_mut, CRISPR, 'mut')], axis=1)

C:\Users\justi\AppData\Local\Temp\ipykernel_11112\3678128755.py:51: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  temp = temp.applymap(np.log)
C:\Users\justi\AppData\Local\Temp\ipykernel_11112\3678128755.py:51: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  temp = temp.applymap(np.log)
C:\Users\justi\AppData\Local\Temp\ipykernel_11112\3678128755.py:51: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  temp = temp.applymap(np.log)
C:\Users\justi\AppData\Local\Temp\ipykernel_11112\3678128755.py:51: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  temp = temp.applymap(np.log)
C:\Users\justi\AppData\Local\Temp\ipykernel_11112\3678128755.py:51: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  temp = temp.applymap(np.log)
C:\Users\justi\AppData\Local\Temp\ipykernel_11112\3678128755.py:51: FutureWarning: DataFrame.ap

In [85]:
dataset.dropna(inplace=True)
dataset

,gene name,gene_exp.copy_num.0,gene_exp.copy_num.1,gene_exp.copy_num.2,gene_exp.copy_num.3,gene_exp.copy_num.4,gene_exp.copy_num.5,gene_exp.copy_num.6,gene_exp.copy_num.7,gene_exp.copy_num.8,...,gene_mut.CRISPR.4,gene_mut.CRISPR.5,gene_mut.CRISPR.6,gene_mut.CRISPR.7,gene_mut.CRISPR.8,gene_mut.CRISPR.9,gene_mut.CRISPR.10,gene_mut.CRISPR.11,gene_mut.CRISPR.12,gene_mut.CRISPR.13
0,A1BG,-7.165493,-7.165493,-7.165493,-7.165493,-7.165493,-7.165493,-7.165493,-7.111943,-5.179541,...,-1.782302,-5.072673,-0.478568,-3.791168,-1.781793,-4.861424,-4.888588,-6.573848,-6.381836,-6.671959
2,A1CF,-7.165493,-7.165493,-7.165493,-7.165493,-7.165493,-7.165493,-7.165493,-7.165489,-7.101136,...,-1.794366,-4.684001,-0.500070,-3.278549,-1.801564,-4.504519,-4.994704,-6.507721,-6.648477,-6.671731
3,A2M,-7.165493,-7.163255,-6.911970,-6.646370,-7.136520,-7.165470,-7.165493,-7.165456,-6.973452,...,-1.853513,-4.366357,-0.476633,-3.151028,-1.880186,-4.388499,-5.019282,-6.482807,-6.519536,-6.671790
5,A2ML1,-7.165493,-7.164623,-6.992000,-6.602891,-7.111818,-7.165420,-7.165493,-7.165490,-7.117275,...,-1.847606,-3.983835,-0.531421,-2.807655,-1.872339,-4.097288,-4.847229,-6.525058,-6.164724,-6.671956
8,A4GALT,-7.165493,-7.165479,-7.140712,-6.429531,-6.142751,-7.062264,-7.165326,-7.157518,-5.725482,...,-1.736648,-5.568323,-0.471883,-4.612959,-1.740518,-5.543676,-4.816489,-6.636474,-6.357813,-6.672016
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28545,ZXDC,-7.165493,-7.165493,-7.165493,-7.165493,-7.165493,-7.165493,-7.165493,-7.157736,-6.636592,...,-1.763331,-4.633987,-0.512129,-3.676785,-1.744760,-4.629632,-4.915740,-6.470520,-5.989535,-6.671743
28546,ZYG11A,-7.165493,-7.165493,-7.165493,-7.165493,-7.165493,-7.165493,-7.165493,-7.165424,-6.952662,...,-1.753182,-4.737137,-0.495659,-3.529987,-1.802538,-4.511773,-5.026294,-6.502216,-5.977413,-6.671866
28548,ZYX,-7.165493,-7.165493,-7.165311,-7.080693,-6.586422,-7.045641,-7.165115,-7.150132,-6.260137,...,-1.744461,-5.071845,-0.523929,-3.823703,-1.670068,-4.936807,-5.122723,-6.600936,-6.665583,-6.671997
28549,ZZEF1,-7.165493,-7.165132,-7.007078,-6.226425,-6.925665,-7.164591,-7.165493,-7.159180,-5.986186,...,-1.738894,-3.969380,-0.629756,-2.763245,-1.758880,-3.873473,-4.495934,-6.170050,-5.960955,-6.669458


In [86]:
dataset.to_csv(r"C:\Users\justi\Coding\Coding_Project\Han Xu\Trimmed data\dataset_trimmed_v3.csv", index=False)